# 2c, $[X_f, X_g] = X_{\{f, g\}}$

**Problem (c).** In the same symplectic setup ($d\omega = 0$, $\iota_{X_h}\omega = dh$, $\{f,g\} := \pi(df, dg)$), the Lie bracket of two Hamiltonian vector fields is again Hamiltonian, and the corresponding function is the Poisson bracket:

$$
[X_f, X_g] = X_{\{f, g\}}.
$$

This is an **operator-level** equality, the identification of two vector fields. Since the system currently has no alternating $p$-form evaluation node, we cannot make the engine equate two vector fields directly (see Phase 12). For this reason we follow a **two-step strategy**:

1. **Via the engine**, show that the action of both sides on $\omega$ via $\iota$ is the same: $\iota_{[X_f, X_g]}\omega = d(X_f(g))$.
2. **In markdown**, invoke non-degeneracy and the result of (b) to close the operator-level conclusion $[X_f, X_g] = X_{\{f, g\}}$.

Since $\omega$ is non-degenerate, a Hamiltonian vector field is uniquely determined by its $\omega$-contraction; the engine closes the first step of this notebook, the second step is an operator-level logical consequence.

## Strategy, element-level target

The identity we start from is Cartan calculus's **$\iota$-$L$ commutation relation** (a.k.a. `lie_iota`):

$$
\iota_{[X, Y]} = L_X \iota_Y - \iota_Y L_X.
$$

Take $(X, Y) = (X_f, X_g)$ and apply both sides to $\omega$:

$$
\iota_{[X_f, X_g]}\omega = L_{X_f}(\iota_{X_g}\omega) - \iota_{X_g}(L_{X_f}\omega).
$$

On the right-hand side:

| Piece | Reduction | Source |
|---|---|---|
| $\iota_{X_g}\omega = dg$ | Hamiltonian defining relation (g) | Axiom A3 |
| $L_{X_f}(dg) \rightsquigarrow d(X_f(g))$ | Cartan magic + pairing + $d^2 = 0$ | Built-in |
| $L_{X_f}\omega \rightsquigarrow 0$ | Cartan magic + $\iota_{X_f}\omega = df$ + $d\omega = 0$ | 2a itself |
| $\iota_{X_g}(0) = 0$ | $\iota_X$ on 0-forms | Built-in |

These three axioms (A3, A4, A8) + the `lie_iota` specialization (A6) suffice for the engine to close the chain. Goal: $\iota_{[X_f, X_g]}\omega = d(X_f(g))$. Note: the right-hand side is equivalent to $d\{f,g\}$ via the result of **(b)** $\{f, g\} = X_f(g)$.

In [1]:
# Make jacopy importable when the notebook is opened directly.
try:
    import jacopy  # noqa: F401
except ModuleNotFoundError:
    import sys
    from pathlib import Path
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "jacopy" / "__init__.py").is_file():
            sys.path.insert(0, str(candidate))
            break
    import jacopy  # noqa: F401

## 1. Setup

- $f, g$, functions (0-forms).
- $\omega$, symplectic 2-form (Graded(2)).
- $X_f, X_g$, Hamiltonian vector fields (degree-0 `Derivation`).
- $L_{X_f}$, Cartan-mode Lie derivative, produced by the call `lie_derivative(X_f)` (bound to the engine via `LieDerivativeCartanDefinition`).
- $[X_f, X_g]$ is built **explicitly** as `Sum(Product(X_f, X_g), Neg(Product(X_g, X_f)))`. Since both fields are degree-0, their graded commutator is even-signed ($AB - BA$), equivalent to the result of `Commutator.expand()`.
- $\iota_{[X_f, X_g]}$, `interior(lie_bracket, name="ι_[X_f,X_g]")`; stores its content as the `vector_field` attribute of `InteriorProduct`.

In [2]:
from jacopy.algebra.derivation import Act, Derivation
from jacopy.calculus.exterior_d import d, ExteriorDerivative
from jacopy.calculus.interior import interior, InteriorProduct
from jacopy.calculus.lie_derivative import lie_derivative
from jacopy.core.expr import Expr, Integer, Neg, Product, Sum, Symbol
from jacopy.core.properties import Graded
from jacopy.core.registry import PropertyRegistry
from jacopy.proof.expansion import Definition, default_engine
from jacopy.proof.strategies import ExpandAndSimplify

reg = PropertyRegistry()

f = Symbol("f")
g = Symbol("g")
reg.declare(f, Graded(degree=0))
reg.declare(g, Graded(degree=0))

omega = Symbol("ω")
reg.declare(omega, Graded(degree=2))

X_f = Derivation("X_f", degree=0)
X_g = Derivation("X_g", degree=0)
L_Xf = lie_derivative(X_f)  # cartan mode (default)

# [X_f, X_g] as an explicit Expr (both degree 0 → even parity).
lie_bracket = Sum(Product(X_f, X_g), Neg(Product(X_g, X_f)))
iota_bracket = interior(lie_bracket, name="ι_[X_f,X_g]")

print("f, g           :", f, g)
print("ω              :", omega, " degree =", reg.get(omega, Graded).degree)
print("X_f, X_g       :", X_f, X_g)
print("L_{X_f}        :", L_Xf)
print("[X_f, X_g]     :", lie_bracket)
print("ι_{[X_f,X_g]}  :", iota_bracket)

f, g           : f g
ω              : ω  degree = 2
X_f, X_g       : X_f X_g
L_{X_f}        : L_X_f
[X_f, X_g]     : ((X_f * X_g) + (-(X_g * X_f)))
ι_{[X_f,X_g]}  : ι_[X_f,X_g]


## 2. Problem axioms

We register four `Definition`s:

- **A6, `lie_iota` specialization**, $\iota_{[X_f, X_g]}(\omega) = L_{X_f}(\iota_{X_g}\omega) - \iota_{X_g}(L_{X_f}\omega)$.
  `CartanCalculus.lie_iota` is the (matchable) form of the general operator-level identity; here we just pin it on its action on $\omega$ at the element level so the engine can reach it bottom-up.
- **A8, $d\omega = 0$**, symplectic closedness (as in 2a).
- **A4, $\iota_{X_f}\omega = df$**, Hamiltonian defining relation (f).
- **A3, $\iota_{X_g}\omega = dg$**, Hamiltonian defining relation (g).

A6 may look axiomatic, it is a theorem in the Cartan calculus infrastructure; `CartanCalculus.verify("lie_iota", ...)` provides the worked-out form. In this notebook we add it as a shortcut axiom; an "unrolled" version can be opened later with `foundational` mode if needed.

In [3]:
class LieIotaOnOmega(Definition):
    """A6 (specialized): ι_{[X_f,X_g]}(ω) = L_{X_f}(ι_{X_g}ω) - ι_{X_g}(L_{X_f}ω)."""
    name = "ι_{[X_f,X_g]}(ω) = L_Xf ι_Xg ω - ι_Xg L_Xf ω"

    def matches(self, expr):
        if not isinstance(expr, Act):
            return False
        op = expr.op
        if not isinstance(op, InteriorProduct):
            return False
        if op.vector_field != lie_bracket:
            return False
        return expr.arg == omega

    def rewrite(self, expr):
        z = expr.arg
        return Sum(
            Act(L_Xf, Act(interior(X_g), z)),
            Neg(Act(interior(X_g), Act(L_Xf, z))),
        )


class DOmegaClosed(Definition):
    """A8: dω = 0."""
    name = "dω = 0 (symplectic)"

    def matches(self, expr):
        return (
            isinstance(expr, Act)
            and isinstance(expr.op, ExteriorDerivative)
            and expr.op == d
            and expr.arg == omega
        )

    def rewrite(self, expr):
        return Integer(0)


class IotaXfOmegaIsDf(Definition):
    """A4: ι_{X_f} ω = df."""
    name = "ι_{X_f} ω = df"

    def matches(self, expr):
        if not isinstance(expr, Act):
            return False
        if not isinstance(expr.op, InteriorProduct):
            return False
        if expr.op.vector_field != X_f:
            return False
        return expr.arg == omega

    def rewrite(self, expr):
        return Act(d, f)


class IotaXgOmegaIsDg(Definition):
    """A3: ι_{X_g} ω = dg."""
    name = "ι_{X_g} ω = dg"

    def matches(self, expr):
        if not isinstance(expr, Act):
            return False
        if not isinstance(expr.op, InteriorProduct):
            return False
        if expr.op.vector_field != X_g:
            return False
        return expr.arg == omega

    def rewrite(self, expr):
        return Act(d, g)


for axiom in (LieIotaOnOmega, DOmegaClosed, IotaXfOmegaIsDf, IotaXgOmegaIsDg):
    print("-", axiom.name)

- ι_{[X_f,X_g]}(ω) = L_Xf ι_Xg ω - ι_Xg L_Xf ω
- dω = 0 (symplectic)
- ι_{X_f} ω = df
- ι_{X_g} ω = dg


## 3. Engine

`default_engine` already carries Cartan magic, $d^2=0$, pairing ($\iota_X(df) = X(f)$), and $\iota_X(f)=0$. We add the four problem axioms on top.

In [4]:
engine = default_engine(registry=reg, d_squared_mode="axiom")
engine.register(LieIotaOnOmega())
engine.register(DOmegaClosed())
engine.register(IotaXfOmegaIsDf())
engine.register(IotaXgOmegaIsDg())

print(f"engine carries {len(engine.definitions)} definitions")
for defn in engine.definitions:
    print(" -", defn.name)

engine carries 12 definitions
 - L_X := d∘ι_X + ι_X∘d (Cartan definition)
 - L_X(f) = X(f) on 0-forms (flow)
 - L_X ∘ d = d ∘ L_X (flow)
 - Act linearity: (A + B)(x) = A(x) + B(x)
 - d² = 0
 - ι_X ∘ ι_X = 0
 - ι_X(f) = 0 on 0-forms
 - ι_X(df) = X(f)
 - ι_{[X_f,X_g]}(ω) = L_Xf ι_Xg ω - ι_Xg L_Xf ω
 - dω = 0 (symplectic)
 - ι_{X_f} ω = df
 - ι_{X_g} ω = dg


## 4. Goal and proof

**Element-level goal** (the engine will close):

$$
\iota_{[X_f, X_g]}\omega = d(X_f(g)).
$$

The right-hand side $d(X_f(g))$, using the result of (b) $\{f, g\} = X_f(g)$, equals $d\{f, g\}$. Thus, once we close this chain, $\iota_{[X_f, X_g]}\omega = d\{f, g\}$; that is, the $\omega$-contraction of $[X_f, X_g]$ is the same as the $\omega$-contraction of the Hamiltonian of $\{f, g\}$.

In [5]:
lhs = Act(iota_bracket, omega)
rhs = Act(d, Act(X_f, g))

print("LHS:", lhs)
print("RHS:", rhs)

chain = ExpandAndSimplify().prove(
    lhs, rhs, registry=reg, engine=engine
)
print(f"\nCLOSED, {len(chain)} steps.")

LHS: ι_[X_f,X_g](ω)
RHS: d(X_f(g))

CLOSED, 15 steps.


## 5. Proof chain, LaTeX

In [6]:
from jacopy.display.jupyter import display_chain

display_chain(chain)

{\allowdisplaybreaks\scriptsize
\begin{gather*}
\iota_{[X_{f,X_g]}}\!\left(\omega\right) \to L_{X_f}\!\left(\iota_{X_g}\!\left(\omega\right)\right) - \iota_{X_g}\!\left(L_{X_f}\!\left(\omega\right)\right) \quad \text{[\ensuremath{\iota}\_{[X\_f,X\_g]}(\ensuremath{\omega}) = L\_Xf \ensuremath{\iota}\_Xg \ensuremath{\omega} - \ensuremath{\iota}\_Xg L\_Xf \ensuremath{\omega}]\,(axiom)} \\
\iota_{X_g}\!\left(\omega\right) \to d\!\left(g\right) \quad \text{[\ensuremath{\iota}\_{X\_g} \ensuremath{\omega} = dg]\,(axiom)} \\
L_{X_f}\!\left(d\!\left(g\right)\right) \to \left(d \, \iota_{X_f}\right)\!\left(d\!\left(g\right)\right) + \left(\iota_{X_f} \, d\right)\!\left(d\!\left(g\right)\right) \quad \text{[L\_X := d\ensuremath{\circ}\ensuremath{\iota}\_X + \ensuremath{\iota}\_X\ensuremath{\circ}d (Cartan definition)]\,(axiom)} \\
L_{X_f}\!\left(\omega\right) \to \left(d \, \iota_{X_f}\right)\!\left(\omega\right) + \left(\iota_{X_f} \, d\right)\!\left(\omega\right) \quad \text{[L\_X := d\ensuremath{\circ}\ensuremath{\iota}\_X + \ensuremath{\iota}\_X\ensuremath{\circ}d (Cartan definition)]\,(axiom)} \\
\left(\left(\left(d \, \iota_{X_f}\right)\!\left(d\!\left(g\right)\right) + \left(\iota_{X_f} \, d\right)\!\left(d\!\left(g\right)\right)\right) - \iota_{X_g}\!\left(\left(d \, \iota_{X_f}\right)\!\left(\omega\right) + \left(\iota_{X_f} \, d\right)\!\left(\omega\right)\right)\right) - d\!\left(X_f\!\left(g\right)\right) \to \left(\left(d\!\left(\iota_{X_f}\!\left(d\!\left(g\right)\right)\right) + \iota_{X_f}\!\left(d\!\left(d\!\left(g\right)\right)\right)\right) - \left(\iota_{X_g}\!\left(d\!\left(\iota_{X_f}\!\left(\omega\right)\right)\right) + \iota_{X_g}\!\left(\iota_{X_f}\!\left(d\!\left(\omega\right)\right)\right)\right)\right) - d\!\left(X_f\!\left(g\right)\right) \quad \text{[product-rule]}\;\text{--- graded Leibniz + linearity} \\
\iota_{X_f}\!\left(d\!\left(g\right)\right) \to X_f\!\left(g\right) \quad \text{[\ensuremath{\iota}\_X(df) = X(f)]\,(axiom)} \\
d\!\left(d\!\left(g\right)\right) \to 0 \quad \text{[d² = 0]\,(axiom)} \\
\iota_{X_f}\!\left(0\right) \to 0 \quad \text{[\ensuremath{\iota}\_X(f) = 0 on 0-forms]\,(axiom)} \\
\iota_{X_f}\!\left(\omega\right) \to d\!\left(f\right) \quad \text{[\ensuremath{\iota}\_{X\_f} \ensuremath{\omega} = df]\,(axiom)} \\
d\!\left(d\!\left(f\right)\right) \to 0 \quad \text{[d² = 0]\,(axiom)} \\
\iota_{X_g}\!\left(0\right) \to 0 \quad \text{[\ensuremath{\iota}\_X(f) = 0 on 0-forms]\,(axiom)} \\
d\!\left(\omega\right) \to 0 \quad \text{[d\ensuremath{\omega} = 0 (symplectic)]\,(axiom)} \\
\iota_{X_f}\!\left(0\right) \to 0 \quad \text{[\ensuremath{\iota}\_X(f) = 0 on 0-forms]\,(axiom)} \\
\iota_{X_g}\!\left(0\right) \to 0 \quad \text{[\ensuremath{\iota}\_X(f) = 0 on 0-forms]\,(axiom)} \\
\left(\left(d\!\left(X_f\!\left(g\right)\right) + 0\right) - \left(0 + 0\right)\right) - d\!\left(X_f\!\left(g\right)\right) \to 0 \quad \text{[simplify]}\;\text{--- canonical-form pipeline}
\end{gather*}
}

## 6. Step by step

The flow of the chain:

1. **A6 `lie_iota`**, LHS opens to `Sum(L_Xf ι_Xg ω, -ι_Xg L_Xf ω)`.
2. **A3**, $\iota_{X_g}\omega$ in the first term drops to $dg$.
3. **Cartan magic (L_Xf d(g))**, first term $L_{X_f}(dg)$ opens via Cartan to $(d\iota_{X_f} + \iota_{X_f} d)(dg)$.
4. **Cartan magic (L_Xf ω)**, second term $L_{X_f}\omega$ opens via Cartan.
5. **product-rule**, composition Acts reduce to element-level form.
6-8. **Pairing, $d^2=0$, $\iota_X(0)=0$**, the first term converges to $d(X_f(g))$.
9-14. **A4, $d^2=0$, A8, $\iota_X(0)=0$**, the second term cascades to $0$.
15. **simplify**, canonical-form closure: $d(X_f(g)) + 0 + (-0) + (-d(X_f(g))) \to 0$.

In [7]:
for i, step in enumerate(chain.steps, 1):
    tag = f"[{step.provenance_tag}]" if step.provenance_tag else ""
    print(f"[{i}] {step.rule} {tag}")
    print(f"    {step.before}")
    print(f" -> {step.after}")
    print()

[1] ι_{[X_f,X_g]}(ω) = L_Xf ι_Xg ω - ι_Xg L_Xf ω [axiom]
    ι_[X_f,X_g](ω)
 -> (L_X_f(ι_X_g(ω)) + (-ι_X_g(L_X_f(ω))))

[2] ι_{X_g} ω = dg [axiom]
    ι_X_g(ω)
 -> d(g)

[3] L_X := d∘ι_X + ι_X∘d (Cartan definition) [axiom]
    L_X_f(d(g))
 -> ((d * ι_X_f)(d(g)) + (ι_X_f * d)(d(g)))

[4] L_X := d∘ι_X + ι_X∘d (Cartan definition) [axiom]
    L_X_f(ω)
 -> ((d * ι_X_f)(ω) + (ι_X_f * d)(ω))

[5] product-rule 
    ((((d * ι_X_f)(d(g)) + (ι_X_f * d)(d(g))) + (-ι_X_g(((d * ι_X_f)(ω) + (ι_X_f * d)(ω))))) + (-d(X_f(g))))
 -> (((d(ι_X_f(d(g))) + ι_X_f(d(d(g)))) + (-(ι_X_g(d(ι_X_f(ω))) + ι_X_g(ι_X_f(d(ω)))))) + (-d(X_f(g))))

[6] ι_X(df) = X(f) [axiom]
    ι_X_f(d(g))
 -> X_f(g)

[7] d² = 0 [axiom]
    d(d(g))
 -> 0

[8] ι_X(f) = 0 on 0-forms [axiom]
    ι_X_f(0)
 -> 0

[9] ι_{X_f} ω = df [axiom]
    ι_X_f(ω)
 -> d(f)

[10] d² = 0 [axiom]
    d(d(f))
 -> 0

[11] ι_X(f) = 0 on 0-forms [axiom]
    ι_X_g(0)
 -> 0

[12] dω = 0 (symplectic) [axiom]
    d(ω)
 -> 0

[13] ι_X(f) = 0 on 0-forms [axiom]
    

## 7. Operator-level closure, non-degeneracy

The engine chain gave us the **element-level** equality:

$$
\iota_{[X_f, X_g]}\omega = d(X_f(g)). \qquad(\star)
$$

From (b), $\{f, g\} = X_f(g)$, so $d\{f, g\} = d(X_f(g))$. The definition of $X_{\{f, g\}}$, the Hamiltonian defining relation, gives $\iota_{X_{\{f,g\}}}\omega = d\{f, g\}$. Chaining this with ($\star$) and dropping the common side:

$$
\iota_{[X_f, X_g]}\omega = \iota_{X_{\{f, g\}}}\omega.
$$

$\omega$ is **non-degenerate**, i.e., $\omega^{\flat}: TM \to T^*M$ is fiber-wise bijective. This makes the map $Y \mapsto \iota_Y \omega$ injective: if the $\omega$-contractions of two vector fields are equal, the fields are equal. Therefore

$$
\boxed{\; [X_f, X_g] = X_{\{f, g\}}. \;}
$$

This last "injectivity" step is not currently in the engine, non-degeneracy is an operator-level axiom (a corner of Phase 12's `MusicalCompatibility`) and equating two vector fields directly requires `AlternatingForm`-aware dispatch. Reducing to $\iota$-contraction and invoking non-degeneracy in markdown is the standard pattern for this family until Phase 12 lands.

## Conclusion

- **Closed by the engine**: $\iota_{[X_f, X_g]}\omega = d(X_f(g))$, 15-step chain, four problem axioms + `default_engine` built-ins.
- **Closed in markdown**: operator-level $[X_f, X_g] = X_{\{f, g\}}$ via non-degeneracy + the result of (b).

**Why two stages?** Equating two vector fields requires applying them to p-forms and matching values via dispatch, with comparison contingent on $\omega$ being non-degenerate. From Phase 12's targets:

- `SymplecticProblem` wrapper, will reduce A3, A4, A8, A6 registrations to a single call.
- `MusicalCompatibility` rank-p extension + non-degeneracy property, will close operator-level $[X_f, X_g] = X_{\{f,g\}}$ without a special call.
- Intrinsic `AlternatingForm` node, will get the $\omega(X, Y)$ node working and lift the dispatch where $\omega(Y, \cdot)$ is injective to the engine level.

We track these three items as separate requests across Phase 12.C (ergonomics) + 12.A/B (intrinsic infrastructure); this notebook is a **painpoint example** for those two layers, justifying the list in plan.md §1193-1372.